# Gromov–Wasserstein Distance

The **Gromov–Wasserstein (GW) distance** compares two metric measure spaces $(X, d_X, \mu_X)$ and $(Y, d_Y, \mu_Y)$ without requiring a common embedding. It is defined as:
$$
\mathrm{GW}_2(X, Y) = \min_{\pi \in \Pi(\mu_X, \mu_Y)} \sum_{i,j,k,l} (d_X(x_i, x_k) - d_Y(y_j, y_l))^2 \,\pi_{ij}\,\pi_{kl},
$$
where $\Pi(\mu_X, \mu_Y)$ is the set of **transport plans** (couplings) with marginals $\mu_X$ and $\mu_Y$.

## Interpretation

GW measures how well the **pairwise distances** within $X$ can be matched to those within $Y$ via a coupling. It is invariant to isometric transformations of each space separately — rotations, reflections, reparametrisation.

## Algorithm: alternating optimisation

The GW problem is non-convex. A common approach alternates:
1. **Fix $\pi$**, compute the linearised cost $C^\pi_{ij} = \sum_{kl}(d_X(x_i,x_k) - d_Y(y_j,y_l))^2 \pi_{kl}$.
2. **Update $\pi$** by solving the linear OT problem $\min_\pi \langle C^\pi, \pi \rangle$.

This is equivalent to **projected gradient descent** on the GW loss.

## Entropic regularisation

Adding entropy $-\varepsilon H(\pi)$ to the objective (Sinkhorn-GW) makes each OT step solvable by the Sinkhorn algorithm and smooths the landscape.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from ipywidgets import interact, IntSlider, FloatLogSlider

plt.rcParams['figure.dpi'] = 120

## Sinkhorn solver and GW iteration

We implement a basic Sinkhorn–GW solver with entropic regularisation.

In [ ]:
def sinkhorn(C, a, b, eps, n_iter=100):
    """Entropic OT: min <C,pi> - eps*H(pi), marginals a, b."""
    K = np.exp(-C / eps)
    u = np.ones(len(a))
    for _ in range(n_iter):
        u = a / (K @ (b / (K.T @ u)))
    v = b / (K.T @ u)
    pi = np.diag(u) @ K @ np.diag(v)
    return pi

def gw_cost_matrix(DX, DY, pi):
    """Linearised GW cost: C_ij = sum_{kl} (DX_ik - DY_jl)^2 * pi_kl."""
    n, m = DX.shape[0], DY.shape[0]
    # Expand: (DX_ik - DY_jl)^2 = DX_ik^2 + DY_jl^2 - 2*DX_ik*DY_jl
    A = (DX**2) @ pi.sum(axis=1, keepdims=True) * np.ones((n, m))
    B = np.ones((n, m)) * (pi.sum(axis=0, keepdims=True) @ (DY**2).T)
    C = 2 * DX @ pi @ DY
    return A + B - C

def gw_loss(DX, DY, pi):
    n, m = len(DX), len(DY)
    loss = 0.0
    for i in range(n):
        for k in range(n):
            for j in range(m):
                for l in range(m):
                    loss += (DX[i,k] - DY[j,l])**2 * pi[i,j] * pi[k,l]
    return loss

def run_gw(DX, DY, a, b, eps=0.1, n_outer=20, n_inner=50):
    """Sinkhorn-GW alternating optimisation."""
    n, m = len(a), len(b)
    pi = np.outer(a, b)  # initialise with product measure
    losses = []
    for _ in range(n_outer):
        C = gw_cost_matrix(DX, DY, pi)
        pi = sinkhorn(C, a, b, eps, n_inner)
        # approximate loss
        L = np.sum(gw_cost_matrix(DX, DY, pi) * pi)
        losses.append(L)
    return pi, np.array(losses)

print('GW solver ready.')

## Comparing two 2D curves

We compare a circle and an ellipse as metric measure spaces (with geodesic or Euclidean distances).

In [ ]:
n1, n2 = 30, 30
t1 = np.linspace(0, 2*np.pi, n1, endpoint=False)
t2 = np.linspace(0, 2*np.pi, n2, endpoint=False)

circle = np.column_stack([np.cos(t1), np.sin(t1)])
ellipse = np.column_stack([2*np.cos(t2), 0.5*np.sin(t2)])

DX = cdist(circle, circle)
DY = cdist(ellipse, ellipse)
a = np.ones(n1) / n1
b = np.ones(n2) / n2

pi, losses = run_gw(DX, DY, a, b, eps=0.05, n_outer=25)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Spaces
axes[0].scatter(circle[:,0], circle[:,1], c=np.arange(n1), cmap='hsv', s=60, zorder=5)
axes[0].scatter(ellipse[:,0]+5, ellipse[:,1], c=np.arange(n2), cmap='hsv', s=60, zorder=5)
# Draw correspondence (top k)
for i in range(n1):
    j = pi[i].argmax()
    alpha = min(1.0, pi[i,j] * n1 * 3)
    axes[0].plot([circle[i,0], ellipse[j,0]+5], [circle[i,1], ellipse[j,1]], 'k-', lw=0.5, alpha=alpha)
axes[0].set_aspect('equal'); axes[0].axis('off')
axes[0].set_title('GW matching: circle ↔ ellipse')

# Transport plan
im = axes[1].imshow(pi, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[1].set_xlabel('ellipse index'); axes[1].set_ylabel('circle index')
axes[1].set_title('Transport plan $\\pi$')

# Convergence
axes[2].semilogy(losses, 'royalblue', lw=2)
axes[2].set_xlabel('outer iteration'); axes[2].set_ylabel('GW loss')
axes[2].set_title('Convergence'); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Comparing shapes: rotation invariance

GW is invariant to rotation. We rotate the second shape by 90°, 180° and verify the GW distance is unchanged.

In [ ]:
# Square and rotated square
n_sq = 20
square_pts = []
for side in range(4):
    s = np.linspace(0, 1, n_sq//4, endpoint=False)
    if side == 0: square_pts.extend(zip(s, np.zeros_like(s)))
    elif side == 1: square_pts.extend(zip(np.ones_like(s), s))
    elif side == 2: square_pts.extend(zip(1-s, np.ones_like(s)))
    else: square_pts.extend(zip(np.zeros_like(s), 1-s))
square = np.array(square_pts) - 0.5

gw_vals = []
angles = np.linspace(0, 2*np.pi, 20)
DX_sq = cdist(square, square)
a_sq = np.ones(n_sq)/n_sq

for angle in angles:
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    square_rot = square @ R.T
    DY_sq = cdist(square_rot, square_rot)
    pi_sq, _ = run_gw(DX_sq, DY_sq, a_sq, a_sq, eps=0.05, n_outer=15)
    gw_vals.append(np.sum(gw_cost_matrix(DX_sq, DY_sq, pi_sq) * pi_sq))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.degrees(angles), gw_vals, 'seagreen', lw=2, marker='o', ms=5)
ax.set_xlabel('rotation angle (degrees)'); ax.set_ylabel('GW loss')
ax.set_title('GW(square, rotated square): near-invariant to rotation')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Interactive: regularisation $\varepsilon$

In [ ]:
def show_gw(log_eps=-1.5, n_outer=20):
    eps_i = 10**log_eps
    pi_i, losses_i = run_gw(DX, DY, a, b, eps=eps_i, n_outer=n_outer)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    # Matching
    for i in range(n1):
        j = pi_i[i].argmax()
        alpha = min(1.0, pi_i[i,j] * n1 * 3)
        axes[0].plot([circle[i,0], ellipse[j,0]+5], [circle[i,1], ellipse[j,1]], 'k-', lw=0.5, alpha=alpha)
    axes[0].scatter(circle[:,0], circle[:,1], c=np.arange(n1), cmap='hsv', s=60, zorder=5)
    axes[0].scatter(ellipse[:,0]+5, ellipse[:,1], c=np.arange(n2), cmap='hsv', s=60, zorder=5)
    axes[0].set_aspect('equal'); axes[0].axis('off')
    axes[0].set_title(fr'GW matching ($\varepsilon={eps_i:.4f}$)')
    im = axes[1].imshow(pi_i, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    axes[1].set_title('Transport plan')
    plt.tight_layout(); plt.show()

interact(show_gw,
         log_eps=FloatLogSlider(value=-1.5, min=-3.0, max=0.0, step=0.25, description='$\\log_{10}\\varepsilon$'),
         n_outer=IntSlider(value=20, min=5, max=50, step=5, description='iters'));

## Bibliographical resources

- Gromov, M. (1999). *Metric Structures for Riemannian and Non-Riemannian Spaces*. Birkhäuser.
- Mémoli, F. (2011). Gromov–Wasserstein distances and the metric approach to object matching. *Foundations of Computational Mathematics*, 11(4), 417–487.
- Peyré, G., Cuturi, M. and Solomon, J. (2016). Gromov-Wasserstein averaging of kernel and distance matrices. *ICML*, 2664–2672.
- Vayer, T., Chapel, L., Flamary, R., Tavenard, R. and Courty, N. (2019). Fused Gromov-Wasserstein distance. arXiv:1811.02834.
- Peyré, G. and Cuturi, M. (2019). Computational optimal transport. *Foundations and Trends in Machine Learning*, 11(5–6), 355–607.